In [2]:
# 1. @chronometrer
from functools import wraps, lru_cache
from time import perf_counter, sleep
def chronometrer(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        debut = perf_counter()
        try:
            return fonction(*args, **kwargs)
        finally:
            duree = perf_counter() - debut
            print(f"{fonction.__name__} a pris {duree:.2f} s")
    return enveloppe
@chronometrer
def lente():
    sleep(1)
    return "Terminé"

print("1.", lente())

lente a pris 1.00 s
1. Terminé


In [6]:
# 2. @majuscules
def majuscules(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        resultat = fonction(*args, **kwargs)
        return resultat.upper() if isinstance(resultat, str) else resultat
    return enveloppe
@majuscules
def bonjour():
    return "bonjour"
print("Bonjour =", bonjour())

Bonjour = BONJOUR


In [8]:
# 3. @compter_appels
def compter_appels(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        enveloppe.appels += 1
        return fonction(*args, **kwargs)
    enveloppe.appels = 0
    return enveloppe
@compter_appels
def addition(a, b):
    return a + b
addition(1, 2)
addition(3, 4)
addition(5, 6)
print("Nombre d'appels :", addition.appels)

Nombre d'appels : 3


In [12]:
# 4. @verifier_positifs
def verifier_positifs(fonction):
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        valeurs = list(args) + list(kwargs.values())
        for valeur in valeurs:
            if isinstance(valeur, (int, float)) and valeur < 0:
                raise ValueError("Les arguments numériques doivent être positifs ou nuls.")
        return fonction(*args, **kwargs)
    return enveloppe

@verifier_positifs
def surface(longueur, largeur):
    return longueur * largeur

print("Surface :", surface(5, 3))
try:
    surface(-5, 3)
except ValueError as erreur:
    print("Erreur :", erreur)

Surface : 15
Erreur : Les arguments numériques doivent être positifs ou nuls.


In [15]:
# 5. @chronometrer(unite="ms") : décorateur avec paramètre
def chronometrer_unite(unite="s"):
    if unite not in ("s", "ms"):
        raise ValueError("L'unité doit être 's' ou 'ms'.")
    def decorateur(fonction):
        @wraps(fonction)
        def enveloppe(*args, **kwargs):
            debut = perf_counter()
            try:
                return fonction(*args, **kwargs)
            finally:
                duree = perf_counter() - debut
                if unite == "ms":
                    print(f"{fonction.__name__} a pris {duree * 1000:.2f} ms")
                else:
                    print(f"{fonction.__name__} a pris {duree:.2f} s")
        return enveloppe
    return decorateur
@chronometrer_unite(unite="ms")
def calcul():
   return sum(range(10000))
print("Résultat :", calcul())

calcul a pris 0.77 ms
Résultat : 49995000


In [ ]:
# 6. @memoriser avec les arguments positionnels et nommés dans la clé
def memoriser(fonction):
    cache = {}
    @wraps(fonction)
    def enveloppe(*args, **kwargs):
        cle = (args, tuple(sorted(kwargs.items())))
        if cle not in cache:
            cache[cle] = fonction(*args, **kwargs)
        return cache[cle]
    enveloppe.cache = cache
    return enveloppe
@chronometrer
@memoriser
def fibonacci_sans_lru(n):
    if n < 2:
        return n
    return fibonacci_sans_lru(n - 1) + fibonacci_sans_lru(n - 2)
@chronometrer
def fibonacci_naif(n):
    if n < 2:
        return n
    return fibonacci_naif(n - 1) + fibonacci_naif(n - 2)
@chronometrer
@lru_cache(maxsize=None)
def fibonacci_lru(n):
    if n < 2:
        return n
    return fibonacci_lru(n - 1) + fibonacci_lru(n - 2)
print("Fibonacci avec mémorisation")
print("Avec @memoriser :", fibonacci_sans_lru(30))
print("Avec @lru_cache :", fibonacci_lru(30))

In [ ]:
# Le calcul naïf de fibonacci(30) peut prendre quelques secondes.
print("Sans cache :", fibonacci_naif(30))

In [19]:
# 7. Empilement : comparer les deux ordres
@chronometrer
@compter_appels
def fonction_a():
    return "A"
@compter_appels
@chronometrer
def fonction_b():
    return "B"
print("\n7. Empilement")
print(fonction_a(), "| appels =", fonction_a.appels)
print(fonction_b(), "| appels =", fonction_b.appels)
print("L'ordre change les attributs disponibles et ce que chaque décorateur enveloppe.")


7. Empilement
fonction_a a pris 0.00 s
A | appels = 0
fonction_b a pris 0.00 s
B | appels = 1
L'ordre change les attributs disponibles et ce que chaque décorateur enveloppe.


In [20]:
# 8. MiniFlask amélioré
class MiniFlask:
    def __init__(self):
        self.routes = {}
        self.gestionnaire_404 = lambda: "404 : page introuvable"
    def route(self, chemin):
        def enregistrer(fonction):
            if chemin in self.routes:
                raise ValueError(f"L'URL {chemin} est déjà enregistrée.")
            self.routes[chemin] = fonction
            return fonction
        return enregistrer
    def lister_routes(self):
        print("Routes enregistrées :")
        for chemin in self.routes:
            print(chemin)
    def erreur_404(self, fonction):
        self.gestionnaire_404 = fonction
        return fonction
    def traiter(self, chemin):
        fonction = self.routes.get(chemin)
        return fonction() if fonction else self.gestionnaire_404()
app = MiniFlask()
@app.route("/")
def accueil():
    return "Bonjour !"
@app.route("/contact")
def contact():
    return "Page contact"
@app.erreur_404
def page_introuvable():
    return "Erreur 404 personnalisée : page introuvable."
print("MiniFlask")
print(app.traiter("/"))
print(app.traiter("/contact"))
print(app.traiter("/inconnue"))
app.lister_routes()

MiniFlask
Bonjour !
Page contact
Erreur 404 personnalisée : page introuvable.
Routes enregistrées :
/
/contact
